## Split Overmerged Authors — Giant Cross-Script Profiles

Targets ~10 author profiles whose feature combinations are physically impossible: `works_count >= 5,000` AND `raw_author_name` mixes multiple writing systems (Latin + CJK / Cyrillic). Each profile contains works of dozens to thousands of real authors merged under one ID due to identical surname+initial blocking on mixed transliterations.

**Examples:**
- A5109189146 `Mamoru Sato`         346,577 works
- A5070718545 `Marina Gritsenko`     70,100 works
- A5102810576 `Wang, Yi-Ting`        19,066 works
- A5100599435 `Dr. Jiali Wu`         11,066 works (135 distinct raw names)
- A5050750924 `Xiao Hu`               9,128 works (115 distinct raw names)

**Action:** null `author_id` in `work_authors` for ALL works on these profiles. MatchAuthors will re-bin them into the real underlying authors.

**Scope:** ~10 authors / ~493K works.

**Excludes** A9999999999 (deleted-profile tombstone, not an overmerge).

**NOTE on cutoffs:** MatchAuthors has cutoffs (`work_id > 7B`, `created_date >= 2025-12-20`). Older split records will NOT be re-processed automatically — a separate re-matching run with cutoffs relaxed is needed for full cleanup.


### Cell 1: Build candidates table

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.overmerge_split_candidates_giant_profiles AS
WITH per_author AS (
  SELECT
    auth.author.id AS author_url,
    COUNT(*) AS n_works_in_works_table,
    MAX(CASE WHEN auth.raw_author_name RLIKE '[\\u4e00-\\u9fff\\u3040-\\u309f\\u30a0-\\u30ff\\uac00-\\ud7af]' THEN 1 ELSE 0 END) AS has_cjk,
    MAX(CASE WHEN auth.raw_author_name RLIKE '[\\u0400-\\u04ff]' THEN 1 ELSE 0 END) AS has_cyr,
    MAX(CASE WHEN auth.raw_author_name RLIKE '[A-Za-z]'
             AND auth.raw_author_name NOT RLIKE '[\\u4e00-\\u9fff\\u3040-\\u309f\\u30a0-\\u30ff\\uac00-\\ud7af\\u0400-\\u04ff]'
        THEN 1 ELSE 0 END) AS has_lat,
    COUNT(DISTINCT LOWER(REGEXP_REPLACE(auth.raw_author_name, '[^a-zA-Z0-9 ]', ''))) AS n_distinct_norm_names
  FROM openalex.works.openalex_works w
  LATERAL VIEW EXPLODE(w.authorships) t AS auth
  WHERE auth.author.id IS NOT NULL
    AND w.publication_year IS NOT NULL
    AND auth.author.id != 'https://openalex.org/A9999999999'
  GROUP BY auth.author.id
  HAVING COUNT(*) >= 5000
     AND (MAX(CASE WHEN auth.raw_author_name RLIKE '[\\u4e00-\\u9fff\\u3040-\\u309f\\u30a0-\\u30ff\\uac00-\\ud7af]' THEN 1 ELSE 0 END)
        + MAX(CASE WHEN auth.raw_author_name RLIKE '[\\u0400-\\u04ff]' THEN 1 ELSE 0 END)
        + MAX(CASE WHEN auth.raw_author_name RLIKE '[A-Za-z]'
                  AND auth.raw_author_name NOT RLIKE '[\\u4e00-\\u9fff\\u3040-\\u309f\\u30a0-\\u30ff\\uac00-\\ud7af\\u0400-\\u04ff]'
             THEN 1 ELSE 0 END)) >= 2
),
giant_authors AS (
  SELECT
    oa.id AS author_id,
    pa.author_url,
    oa.block_key,
    oa.full_name,
    oa.works_count,
    pa.n_works_in_works_table,
    pa.n_distinct_norm_names,
    pa.has_cjk, pa.has_cyr, pa.has_lat
  FROM per_author pa
  LEFT JOIN openalex.authors.openalex_authors oa
    ON CONCAT('https://openalex.org/A', oa.id) = pa.author_url
  WHERE oa.id IS NOT NULL
)
-- Final candidate rows: every (work_id, author_sequence) for these authors.
SELECT
  wa.work_id,
  wa.author_sequence,
  wa.author_id,
  wa.raw_author_name,
  ga.block_key,
  ga.full_name AS author_full_name,
  ga.works_count AS author_works_count,
  ga.n_works_in_works_table,
  ga.n_distinct_norm_names,
  ga.has_cjk, ga.has_cyr, ga.has_lat
FROM openalex.works.work_authors wa
JOIN giant_authors ga
  ON wa.author_id = ga.author_id
WHERE wa.author_id IS NOT NULL

### Cell 2: Validation statistics

In [ ]:
SELECT
  COUNT(*)                       AS total_candidates,
  COUNT(DISTINCT author_id)      AS distinct_authors,
  COUNT(DISTINCT work_id)        AS distinct_works
FROM openalex.authors.overmerge_split_candidates_giant_profiles

### Cell 3: Per-author summary — verify the list before splitting

In [ ]:
SELECT
  author_id,
  ANY_VALUE(author_full_name)    AS author_full_name,
  ANY_VALUE(block_key)           AS block_key,
  ANY_VALUE(author_works_count)  AS works_count,
  ANY_VALUE(n_works_in_works_table) AS n_works_in_works_table,
  ANY_VALUE(n_distinct_norm_names)  AS n_distinct_norm_names,
  COUNT(*) AS works_to_null
FROM openalex.authors.overmerge_split_candidates_giant_profiles
GROUP BY author_id
ORDER BY works_to_null DESC

### Cell 4: Create audit log (for rollback)

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.overmerge_split_log_giant_profiles AS
SELECT
  work_id,
  author_sequence,
  author_id AS previous_author_id,
  raw_author_name,
  block_key,
  author_full_name,
  current_timestamp() AS split_timestamp
FROM openalex.authors.overmerge_split_candidates_giant_profiles

### Cell 5: Execute the split — null author_id in work_authors

**WARNING:** this nulls ~493K author_ids. Verify cells 2-3 first.

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT DISTINCT work_id, author_sequence
  FROM openalex.authors.overmerge_split_candidates_giant_profiles
) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED THEN
  UPDATE SET
    target.author_id = NULL,
    target.updated_at = current_timestamp()

### Cell 6: Queue affected works for re-sync via UpdateWorkAuthorships

In [ ]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT DISTINCT
  work_id,
  NULL AS curated_ras,
  current_timestamp() AS added_datetime
FROM openalex.authors.overmerge_split_candidates_giant_profiles

### Cell 7: Post-split verification

In [ ]:
SELECT
  COUNT(*) AS nulled_records,
  COUNT(DISTINCT c.work_id) AS works_affected
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_log_giant_profiles c
  ON wa.work_id = c.work_id AND wa.author_sequence = c.author_sequence
WHERE wa.author_id IS NULL

### Cell 8: Outcome tracking — run after MatchAuthors re-processes

In [ ]:
SELECT
  CASE
    WHEN wa.author_id IS NULL THEN 'STILL_UNMATCHED'
    WHEN wa.author_id = log.previous_author_id THEN 'RE_MATCHED_SAME'
    ELSE 'RE_MATCHED_NEW'
  END AS outcome,
  COUNT(*) AS cnt
FROM openalex.works.work_authors wa
JOIN openalex.authors.overmerge_split_log_giant_profiles log
  ON wa.work_id = log.work_id AND wa.author_sequence = log.author_sequence
GROUP BY 1